# Semana 7: redes neuronales para mantenimiento predictivo industrial

En las semanas anteriores se usaron regresión lineal, árboles de decisión y K-Means. Este encuentro presenta las **redes neuronales artificiales** y las aplica a un problema real de la industria: **anticipar fallas en máquinas CNC** para pasar del mantenimiento correctivo (reparar cuando se daña) al **mantenimiento predictivo** (intervenir antes de que se dañe). Al final, la red queda disponible en una **app web de Streamlit** donde se puede probar, validar y usar para responder la pregunta de negocio.

| Paso | Qué se hace | Herramienta |
|---|---|---|
| 0 | **Partir de la pregunta de negocio** | — |
| 1 | Preparar el entorno e importar las clases nuevas | `RedNeuronalMantenimiento`, `FrontendRedNeuronal` |
| 2 | Plantear el caso industrial y la pregunta de negocio | — |
| 3 | Entender cómo funciona una red neuronal (con NumPy, desde cero) | NumPy |
| 4 | Explorar los datos de sensores (AI4I 2020) | pandas, Plotly |
| 5 | Preparar los datos: variables físicas, división y escalado | `preparar()` |
| 6 | Entrenar el perceptrón multicapa | `MLPClassifier` (`entrenar()`) |
| 7 | Evaluar sin caer en la trampa de la exactitud | `evaluar()`, matriz de confusión, ROC, PR |
| 8 | Elegir el umbral de decisión según los **costos** del negocio | `curva_costos()`, `umbral_optimo()` |
| 9 | Responder la pregunta de negocio y generar la orden de trabajo | `respuesta_negocio()`, `priorizar()` |
| 10 | Guardar el modelo | `guardar()` |
| 11 | Crear y probar el frontend en Streamlit | `FrontendRedNeuronal.render()`, `AppTest` |
| 12 | Publicar la app | GitHub + Streamlit Community Cloud |
| 13 | **El tablero responde la pregunta**: monitoreo por fecha y turno | `simular_operacion()`, `resumen_monitoreo()`, `carga_personal()`, `deriva()`, `orden_trabajo()` |

```
sensores de la máquina ──► RedNeuronalMantenimiento ──► probabilidad de falla ──► umbral por costo ──► ¿parar o seguir?
 (temperaturas, rpm,             (MLP: 32 → 16 → 1)                                                         │
  torque, desgaste)                                                                                          ▼
                                        FrontendRedNeuronal.render() ──► app web: probar · validar · decidir
```

## 🎬 Cómo usar este notebook con la animación

La clase se da con la animación **«Redes neuronales en escena»** (9 escenas). Cada sección de este notebook lleva una marca **🎬 Escena N — Momento M**: cuando la animación se pausa al final de un momento, ejecuta las celdas que tienen esa marca y explica el código.

| Escena | Momentos | Secciones del notebook |
|---|---|---|
| **1 · Todo empieza con una pregunta** | Datos sin pregunta · La pregunta · Desarmar la pregunta · Las preguntas hijas | **0** |
| **2 · Cambié una sola cosa** | El costo de esperar · La falla cuesta 17 veces más · Las fallas son combinaciones | **2** |
| **3 · ¿Qué es una red neuronal?** | La neurona · Un ejemplo a mano · Activaciones · El problema XOR | **3.1 – 3.3** |
| **4 · Cómo aprende la red** | El ciclo de cada época · La fuerza bruta no escala · Bajar paso a paso · MLP y sus perillas · Tres cuidados | **3.3 – 3.6** |
| **5 · De los sensores a los datos** | Los datos · Desbalance y modos de falla · ¿Dónde aparecen las fallas? · Variables físicas · Dividir y escalar | **4 – 5** |
| **6 · Entrena y evalúa tu red** | La arquitectura · Entrenamiento · La trampa de la exactitud · Matriz de confusión · ¿Vale la pena la red? · ¿Más neuronas es mejor? | **6 – 7** |
| **7 · El umbral se elige con dinero** | Costo por umbral · El rigor · Si cambian los costos · La respuesta de negocio · ¿Qué fallas anticipa? · Orden de trabajo | **8 – 9** |
| **8 · Del modelo a la herramienta** | Dos clases · La app en Streamlit · Probar y publicar · Límites honestos | **10 – 12** |
| **9 · El tablero responde la pregunta** | Fuente de datos · Periodo y turnos · Indicadores · Personal · Monitoreo de los datos · Orden de trabajo · Cierre | **13** |

La secuencia es la de la animación: **una pregunta de negocio** abre la clase y **el tablero** la cierra respondiendo cada una de las preguntas que se desprenden de ella.

---
# 1 · Preparar el entorno

Instala `streamlit` y `plotly` en el entorno virtual (solo la primera vez). `scikit-learn` ya está en el `requirements.txt` de la raíz.

In [1]:
%pip install -q "streamlit>=1.50" "plotly>=5.24" scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Como en las semanas anteriores, se busca la carpeta `src/clases_po` y se agrega `src` al `sys.path` para importar las clases.

Esta semana hay **dos clases nuevas**:

| Clase | Archivo | Responsabilidad |
|---|---|---|
| `RedNeuronalMantenimiento` | `src/clases_po/red_neuronal.py` | Datos, red neuronal, evaluación, costos, decisión y guardado |
| `FrontendRedNeuronal` | `src/clases_po/frontend_red_neuronal.py` | Figuras y la app web de Streamlit para probar y validar la red |

Separarlas es una buena práctica: el **modelo** no sabe nada de la interfaz y la **interfaz** solo consume los métodos del modelo.

In [2]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from clases_po import RedNeuronalMantenimiento, FrontendRedNeuronal
from clases_po.red_neuronal import cargar_datos_ai4i, MODOS_FALLA

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print("Raíz del proyecto:", ROOT)

d:\IUDigital\2026\q4\machine learning\repositorios\ML_2026Q4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raíz del proyecto: D:\IUDigital\2026\q4\machine learning\repositorios\ML_2026Q4


---
# 0 · Todo empieza con una pregunta

> 🎬 **Escena 1 · Todo empieza con una pregunta** — **Momento 1 · Datos sin pregunta**

Antes de cualquier modelo, mira lo que te entregan: miles de filas con temperaturas, velocidad, torque y desgaste. Puedes graficar, promediar, entrenar diez modelos… y al final nadie en la planta sabrá qué hacer el lunes. **Los datos solos no dicen qué decidir.**

In [3]:
RUTA_DATOS = ROOT / "src" / "sincronicas" / "datos" / "ai4i2020.csv"
df, origen = cargar_datos_ai4i(RUTA_DATOS)
print(f"Te entregan {df.shape[0]:,} filas × {df.shape[1]} columnas ({origen}).")
print("¿Qué decides con esto? Sin una pregunta, nada.")
df.head()

Te entregan 10,000 filas × 14 columnas (archivo local ai4i2020.csv).
¿Qué decides con esto? Sin una pregunta, nada.


,udi,producto_id,tipo,temp_aire_K,temp_proceso_K,velocidad_rpm,torque_Nm,desgaste_min,falla,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


> 🎬 **Escena 1 · Todo empieza con una pregunta** — **Momento 2 · La pregunta**

> ## ¿Qué máquinas detener en el próximo turno para anticipar la mayor cantidad de fallas al menor costo?

Una sola frase de negocio. Pero esta frase **decide todo lo que vamos a construir**.

> 🎬 **Escena 1 · Todo empieza con una pregunta** — **Momento 3 · Desarmar la pregunta**

| Parte de la pregunta | Decisión técnica que obliga | Dónde aparece en el notebook |
|---|---|---|
| **¿Qué máquinas?** | Un puntaje **por máquina**, no un promedio de la planta → probabilidad y ranking | `predecir_proba()`, `priorizar()` |
| **detener** | Una acción **sí o no** → hay que elegir un **umbral** | Sección 8 |
| **en el próximo turno** | Horizonte y frecuencia: el modelo **corre cada turno** con las lecturas nuevas | Sección 13 (tablero) |
| **la mayor cantidad de fallas** | Se mide cuántas fallas se **anticipan** (recall), no la exactitud | Sección 7 |
| **al menor costo** | Hay que ponerle **dinero** a cada error: US$ 10 000 por falla vs. US$ 600 por mantenimiento | Sección 8 |

Nada de esto salió de los datos: **salió de la pregunta**.

> 🎬 **Escena 1 · Todo empieza con una pregunta** — **Momento 4 · Las preguntas hijas**

De la pregunta principal nacen otras. Hoy construimos un sistema que las responde todas; al final de la clase, el **tablero** abre cada candado:

| 🔒 Pregunta hija | Dónde se responde |
|---|---|
| ¿Cuántas fallas anticipamos? | 7 (prueba) y 13 · Momento 3 (tablero) |
| ¿Cuánto ahorramos? | 9 (prueba) y 13 · Momento 3 |
| ¿Cuántas máquinas intervenir por turno? | 9.2 y 13 · Momento 6 |
| ¿Cuántos técnicos necesito? | 13 · Momento 4 |
| ¿Siguen iguales los datos? | 13 · Momento 5 |
| ¿Cuál es la orden de trabajo de hoy? | 13 · Momento 6 |

---
# 2 · El caso industrial

> 🎬 **Escena 2 · Cambié una sola cosa** — **Momento 1 · El costo de esperar a que fallen** · Momento 2 · La falla cuesta 17 veces más · Momento 3 · Las fallas son combinaciones (se ve en la sección 4.2)

**Empresa:** una planta metalmecánica que fabrica piezas en fresadoras CNC, en tres líneas de calidad de producto: **L** (baja, 50 %), **M** (media, 30 %) y **H** (alta, 20 %).

**Situación actual:** el mantenimiento es **correctivo**: la máquina se repara cuando falla. Cada parada no planificada detiene la línea, daña piezas en proceso, obliga a pagar horas extra y retrasa entregas.

**Datos disponibles:** cada máquina tiene sensores que registran 5 variables de operación:

| Variable | Qué mide | Por qué importa |
|---|---|---|
| `temp_aire_K` | Temperatura del ambiente (Kelvin) | Si el aire está caliente, la máquina disipa peor el calor |
| `temp_proceso_K` | Temperatura del proceso de corte | Debe estar ~10 K por encima del aire |
| `velocidad_rpm` | Velocidad de giro del husillo | Con el torque define la **potencia** |
| `torque_Nm` | Fuerza de giro | Mucho torque con herramienta gastada = sobreesfuerzo |
| `desgaste_min` | Minutos de uso de la herramienta | Las herramientas fallan entre los 200 y 240 min |

**Necesidad:** el jefe de mantenimiento quiere saber, **al inicio de cada turno, qué máquinas detener** para hacerles mantenimiento preventivo. No puede parar todas (es caro y frena la producción), pero tampoco quiere que una máquina falle en plena operación.

**Pregunta de negocio:**

> ¿Qué máquinas debemos detener para mantenimiento preventivo en el próximo turno, de modo que anticipemos la mayor cantidad de fallas al menor costo total?

**Supuestos de costo** (se pueden cambiar en la app):

| Evento | Costo | Qué incluye |
|---|---|---|
| Falla no anticipada | US$ 10 000 | Reparación correctiva, piezas dañadas, producción perdida, horas extra |
| Mantenimiento preventivo | US$ 600 | Técnico, repuestos y una parada corta **planificada** |

Una falla cuesta ~17 veces más que un mantenimiento. Por eso conviene parar una máquina aunque la probabilidad de falla no sea del 50 %: basta con que el costo esperado de seguir (`probabilidad × 10 000`) supere el de parar (`600`).

**¿Por qué una red neuronal?** Las fallas no dependen de una sola variable ni de una relación lineal: aparecen cuando **combinaciones** de variables cruzan ciertos límites (poca diferencia de temperatura **y** baja velocidad; mucho desgaste **y** mucho torque). Una red neuronal aprende esas interacciones no lineales a partir de los datos, sin que se le programen las reglas.

In [4]:
COSTO_FALLA, COSTO_MANTENIMIENTO = 10_000, 600
print(f"Una falla no anticipada cuesta {COSTO_FALLA / COSTO_MANTENIMIENTO:.0f} veces más que un mantenimiento preventivo.")
print(f"En el conjunto de prueba hay 51 fallas: sin modelo cuestan US$ {51 * COSTO_FALLA:,}.")

Una falla no anticipada cuesta 17 veces más que un mantenimiento preventivo.
En el conjunto de prueba hay 51 fallas: sin modelo cuestan US$ 510,000.


---
# 3 · ¿Qué es una red neuronal?

> 🎬 **Escena 3 · ¿Qué es una red neuronal?** — **Momento 1 · La neurona**
>
> El ejemplo a mano de la celda siguiente es el **Momento 2 · Un ejemplo a mano**.

## 3.1 La neurona artificial

Una neurona hace dos cosas:

1. **Suma ponderada:** multiplica cada entrada por un **peso** y suma un **sesgo**: $z = w_1x_1 + w_2x_2 + \dots + w_nx_n + b = \mathbf{w}\cdot\mathbf{x} + b$
2. **Activación:** pasa $z$ por una función no lineal $a = f(z)$.

```
 x₁ ──w₁──┐
 x₂ ──w₂──┼──► z = Σ wᵢxᵢ + b ──► a = f(z) ──► salida
 x₃ ──w₃──┘
```

Los **pesos** dicen cuánto importa cada entrada (y en qué sentido); el **sesgo** mueve el punto de activación. *Aprender* es encontrar los pesos y sesgos que mejor predicen.

Ejemplo a mano: una neurona que mira tres sensores (ya estandarizados: 0 = valor promedio, +2 = muy alto) y usa activación sigmoide para dar una probabilidad.

In [5]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

x = np.array([1.8, 2.1, -0.5])        # torque alto, desgaste alto, velocidad algo baja (estandarizados)
w = np.array([1.2, 1.5, -0.8])        # pesos: torque y desgaste suben el riesgo; la velocidad lo baja
b = -4.0                              # sesgo: por defecto la máquina "no falla"

z = w @ x + b
print(f"z = {' + '.join(f'({wi}×{xi})' for wi, xi in zip(w, x))} + ({b}) = {z:.2f}")
print(f"a = sigmoide(z) = {sigmoide(z):.3f}  → probabilidad de falla según esta neurona")

x_normal = np.array([0.0, -0.3, 0.2])
print(f"Máquina normal: z = {w @ x_normal + b:.2f} → probabilidad = {sigmoide(w @ x_normal + b):.3f}")

z = (1.2×1.8) + (1.5×2.1) + (-0.8×-0.5) + (-4.0) = 1.71
a = sigmoide(z) = 0.847  → probabilidad de falla según esta neurona
Máquina normal: z = -4.61 → probabilidad = 0.010


## 3.2 Funciones de activación

> 🎬 **Escena 3 · ¿Qué es una red neuronal?** — **Momento 3 · Activaciones**

Sin la activación, apilar neuronas sería solo una suma de sumas: una función **lineal**. La activación es lo que permite dibujar curvas.

| Activación | Fórmula | Uso típico |
|---|---|---|
| **Sigmoide** | $1/(1+e^{-z})$ | Capa de salida en clasificación binaria (da una probabilidad 0–1) |
| **Tanh** | $\tanh(z)$ | Capas ocultas (salida entre −1 y 1) |
| **ReLU** | $\max(0, z)$ | Capas ocultas, la más usada hoy: simple y entrena rápido |

In [6]:
FrontendRedNeuronal.figura_activaciones()

## 3.3 Una neurona sola no basta: el problema XOR

> 🎬 **Escena 3 · ¿Qué es una red neuronal?** — **Momento 4 · El problema XOR**

XOR (o exclusivo) vale 1 cuando **solo una** de dos entradas es 1. Es el ejemplo clásico de un patrón que **no se separa con una línea recta**, justo como las fallas de las máquinas, que aparecen cuando se combinan condiciones.

Primero, una sola neurona (equivalente a la regresión logística):

In [7]:
from sklearn.linear_model import LogisticRegression

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

una_neurona = LogisticRegression().fit(X_xor, y_xor)
print("Real:          ", y_xor)
print("Una neurona:   ", una_neurona.predict(X_xor), f"→ exactitud {una_neurona.score(X_xor, y_xor):.0%}")

Real:           [0 1 1 0]
Una neurona:    [0 0 0 0] → exactitud 50%


> 🎬 **Escena 4 · Cómo aprende la red** — **Momento 1 · El ciclo de cada época**

Ahora una red con una **capa oculta de 4 neuronas**, programada desde cero con NumPy. El ciclo de entrenamiento tiene 4 pasos que se repiten en cada **época**:

1. **Propagación hacia adelante (forward):** las entradas pasan capa por capa hasta la salida.
2. **Pérdida:** se mide qué tan lejos está la predicción de la realidad (entropía cruzada o *log-loss*).
3. **Retropropagación (backpropagation):** con la regla de la cadena se calcula cuánto contribuyó **cada peso** al error (el gradiente).
4. **Descenso del gradiente:** cada peso se mueve un poco en la dirección que baja la pérdida: $w \leftarrow w - \eta \cdot \frac{\partial \text{pérdida}}{\partial w}$, donde $\eta$ es la **tasa de aprendizaje**.

In [8]:
rng = np.random.default_rng(1)
W1, b1 = rng.normal(0, 1, (2, 4)), np.zeros((1, 4))     # entrada (2) → oculta (4)
W2, b2 = rng.normal(0, 1, (4, 1)), np.zeros((1, 1))     # oculta (4) → salida (1)
tasa = 0.5
Y = y_xor.reshape(-1, 1)
historial = []

for epoca in range(3000):
    # 1) forward
    a1 = np.tanh(X_xor @ W1 + b1)
    a2 = sigmoide(a1 @ W2 + b2)
    # 2) pérdida
    perdida = -np.mean(Y * np.log(a2 + 1e-9) + (1 - Y) * np.log(1 - a2 + 1e-9))
    historial.append(perdida)
    # 3) backpropagation (regla de la cadena)
    dz2 = (a2 - Y) / len(Y)
    dW2, db2 = a1.T @ dz2, dz2.sum(0, keepdims=True)
    dz1 = (dz2 @ W2.T) * (1 - a1 ** 2)                  # derivada de tanh
    dW1, db1 = X_xor.T @ dz1, dz1.sum(0, keepdims=True)
    # 4) descenso del gradiente
    W2 -= tasa * dW2; b2 -= tasa * db2
    W1 -= tasa * dW1; b1 -= tasa * db1

prob = sigmoide(np.tanh(X_xor @ W1 + b1) @ W2 + b2).ravel()
print("Pérdida: época 1 =", round(historial[0], 3), "→ época 3000 =", round(historial[-1], 4))
print("Probabilidades:", prob.round(3))
print("Red de 2 capas:", (prob >= 0.5).astype(int), "→ exactitud", f"{((prob >= 0.5) == y_xor).mean():.0%}")
print("Parámetros aprendidos:", W1.size + b1.size + W2.size + b2.size)

Pérdida: época 1 = 0.671 → época 3000 = 0.0016
Probabilidades: [0.    0.998 0.998 0.003]
Red de 2 capas: [0 1 1 0] → exactitud 100%
Parámetros aprendidos: 17


La capa oculta transforma las entradas en un nuevo espacio donde el problema **sí** se separa. Eso es lo que hace una red neuronal: **aprende sus propias variables intermedias**.


## 3.4 ¿Por qué descenso del gradiente?

> 🎬 **Escena 4 · Cómo aprende la red** — **Momento 2 · La fuerza bruta no escala**

El descenso del gradiente entra en acción cuando la matemática clásica amenaza con **saturar el procesador**. En una regresión lineal existe una respuesta directa, la **ecuación normal**:

$$\mathbf{w} = (X^\top X)^{-1} X^\top \mathbf{y}$$

El problema es la matriz $X^\top X$, de tamaño $p \times p$ (una fila y una columna por variable). Armarla cuesta del orden de $n \cdot p^2$ operaciones e invertirla del orden de $p^3$: si **duplicas las variables, el cálculo se multiplica por cerca de 8**. Con millones de variables, el sistema colapsa.

El descenso del gradiente, en cambio, hace en cada iteración un cálculo que cuesta del orden de $n \cdot p$: **duplicar las variables solo duplica** el trabajo de cada paso.

La celda siguiente lo mide en tu computador: el mismo problema resuelto con la ecuación normal y con 200 pasos de descenso del gradiente, duplicando el número de variables.

In [9]:
import time

def ecuacion_normal(X, y):
    return np.linalg.solve(X.T @ X, X.T @ y)          # arma XᵀX (p × p) y resuelve el sistema

def descenso_gradiente(X, y, eta=0.1, pasos=200):
    w = np.zeros(X.shape[1])
    for _ in range(pasos):
        gradiente = X.T @ (X @ w - y) / len(y)         # costo de cada paso ≈ n · p
        w -= eta * gradiente
    return w

rng_t = np.random.default_rng(0)
n = 4_000
filas = []
for p in [250, 500, 1_000, 2_000]:
    X = rng_t.normal(size=(n, p)) / np.sqrt(p)
    y = X @ rng_t.normal(size=p) + rng_t.normal(0, 0.1, n)
    t0 = time.perf_counter(); ecuacion_normal(X, y); t_normal = time.perf_counter() - t0
    t0 = time.perf_counter(); descenso_gradiente(X, y); t_gd = time.perf_counter() - t0
    filas.append({"variables p": p, "ecuación normal (s)": t_normal, "descenso, 200 pasos (s)": t_gd})

tiempos = pd.DataFrame(filas).set_index("variables p")
tiempos["normal: × al duplicar p"] = tiempos["ecuación normal (s)"] / tiempos["ecuación normal (s)"].shift()
tiempos["descenso: × al duplicar p"] = tiempos["descenso, 200 pasos (s)"] / tiempos["descenso, 200 pasos (s)"].shift()
display(tiempos.round(3))

# Extrapolación a 1 000 000 de variables con el crecimiento teórico de cada método
p0, p1 = 2_000, 1_000_000
t_normal_1M = tiempos.loc[p0, "ecuación normal (s)"] * (p1 / p0) ** 3
t_gd_1M = tiempos.loc[p0, "descenso, 200 pasos (s)"] * (p1 / p0)
print(f"Estimado con p = 1 000 000 variables:")
print(f"  ecuación normal ≈ {t_normal_1M / 86_400 / 365:,.1f} años  (y XᵀX ocuparía {p1**2 * 8 / 1e12:,.0f} TB de memoria)")
print(f"  descenso, 200 pasos ≈ {t_gd_1M / 60:,.0f} minutos")

,ecuación normal (s),"descenso, 200 pasos (s)",normal: × al duplicar p,descenso: × al duplicar p
variables p,,,,
250,0.176,0.080,NaN,NaN
500,0.183,0.085,1.037,1.066
1000,0.261,0.227,1.429,2.674
2000,3.868,0.609,14.832,2.680


Estimado con p = 1 000 000 variables:
  ecuación normal ≈ 15.3 años  (y XᵀX ocuparía 8 TB de memoria)
  descenso, 200 pasos ≈ 5 minutos


**Lectura:** con pocas variables, la ecuación normal todavía gana en tiempo absoluto (NumPy usa álgebra lineal muy optimizada). Lo que importa es **cómo crece**: mira las dos últimas columnas. Al duplicar las variables, la ecuación normal se multiplica cada vez más (tiende a ×8), mientras el descenso solo se duplica. Por eso, al extrapolar a un millón de variables, la fuerza bruta matricial tardaría años y no cabría en memoria, y el descenso termina en minutos. Los números exactos dependen de tu procesador.

> 🎬 **Escena 4 · Cómo aprende la red** — **Momento 3 · Bajar paso a paso**

Abandonamos la respuesta instantánea y pasamos a una **aproximación iterativa** sobre la superficie de costo. En cada ciclo restamos el gradiente del error:

$$w \leftarrow w - \eta \cdot \frac{\partial\, \text{pérdida}}{\partial w}$$

Pero el **tamaño del paso** importa: un salto enorme rebota de lado a lado, y uno minúsculo estanca el avance. A ese regulador lo llamamos **tasa de aprendizaje** ($\eta$). La celda siguiente entrena la misma regresión con tres tasas distintas.

> **Ojo:** en una regresión lineal la superficie de costo es un solo valle y el descenso llega al mínimo global. En una red neuronal hay muchos valles: el descenso llega a **un** mínimo bueno, no necesariamente al global. Adam, la inicialización y la regularización ayudan a caer en uno bueno.

In [10]:
import plotly.graph_objects as go

X = rng_t.normal(size=(2_000, 50)) / np.sqrt(50)
y = X @ rng_t.normal(size=50) + rng_t.normal(0, 0.1, 2_000)

def curva_perdida(eta, pasos=60):
    w, perdidas = np.zeros(X.shape[1]), []
    for _ in range(pasos):
        error = X @ w - y
        perdidas.append(float(np.mean(error ** 2)))
        w -= eta * X.T @ error / len(y)
    return perdidas

fig = go.Figure()
for eta, nombre, color in [(80.0, "η = 80 · rebota y se dispara", "#e63946"),
                           (0.05, "η = 0.05 · casi no avanza", "#8d99ae"),
                           (20.0, "η = 20 · baja al fondo", "#2a9d8f")]:
    fig.add_trace(go.Scatter(y=curva_perdida(eta), name=nombre, line=dict(color=color, width=3)))
fig.update_yaxes(type="log", title="Pérdida (error cuadrático medio, escala log)")
fig.update_xaxes(title="Iteración")
fig.update_layout(title="El tamaño del paso decide si el descenso converge", template="plotly_white", height=380)
fig

El mismo principio entrenó la red XOR de la sección 3.3: su pérdida bajó de **0,671 a 0,0016** en 3 000 épocas.

## 3.5 El perceptrón multicapa (MLP) y sus perillas

> 🎬 **Escena 4 · Cómo aprende la red** — **Momento 4 · MLP y sus perillas**

Para el caso industrial no se programa la red a mano: se usa `MLPClassifier` de scikit-learn, que hace exactamente lo anterior, con más capas y un optimizador mejor (**Adam**).

| Hiperparámetro | En `RedNeuronalMantenimiento` | Qué controla |
|---|---|---|
| `capas_ocultas` | `(32, 16)` | Cuántas capas y cuántas neuronas en cada una (capacidad de la red) |
| `activacion` | `"relu"` | Función de activación de las capas ocultas |
| `tasa_aprendizaje` | `0.001` | Tamaño del paso del descenso del gradiente |
| `alpha` | `0.001` | Regularización L2: castiga pesos grandes para evitar memorizar |
| `max_iter` | `300` | Número de épocas (pasadas completas por los datos) |
| `batch_size` | `128` | Cuántos ejemplos se usan para cada actualización de pesos |


Estas perillas se fijan en la sección 5, en la celda que crea `RedNeuronalMantenimiento(...)`.

## 3.6 Tres cuidados obligatorios con redes neuronales

> 🎬 **Escena 4 · Cómo aprende la red** — **Momento 5 · Tres cuidados**

| Cuidado | Por qué | Cómo se resuelve aquí |
|---|---|---|
| **Escalar** las variables | El torque está en decenas y la velocidad en miles: sin escalar, los pesos de la velocidad dominan y el entrenamiento se vuelve lento e inestable | `StandardScaler` dentro del `Pipeline` |
| **Desbalance** de clases | Solo ~3 % de registros son fallas: la red aprende que "nunca falla" es una buena respuesta | Sobremuestreo de fallas en el entrenamiento y umbral por costo |
| **Sobreajuste** | Con muchos parámetros la red puede memorizar el entrenamiento | Regularización `alpha` y evaluar en datos que nunca vio (validación y prueba) |

---
# 4 · Los datos: AI4I 2020 Predictive Maintenance

> 🎬 **Escena 5 · De los sensores a los datos** — **Momento 1 · Los datos**

El dataset **AI4I 2020** (UCI Machine Learning Repository) tiene **10 000 registros** de máquinas con sus 5 variables de operación, el tipo de producto y si hubo falla. Además indica el **modo de falla**:

| Código | Modo de falla | Cuándo ocurre |
|---|---|---|
| TWF | Desgaste de herramienta | La herramienta se rompe entre 200 y 240 min de uso |
| HDF | Disipación de calor | Diferencia proceso − aire < 8.6 K **y** velocidad < 1 380 rpm |
| PWF | Potencia | Potencia (torque × velocidad angular) < 3 500 W o > 9 000 W |
| OSF | Sobreesfuerzo | Desgaste × torque > 11 000 (L), 12 000 (M) o 13 000 (H) min·Nm |
| RNF | Aleatoria | 0.1 % de probabilidad, sin causa en los datos |

> **Importante:** los modos de falla se usan solo para **analizar** el resultado. La red **no** los recibe como entrada: si los recibiera, conocería la respuesta (fuga de información).

`cargar_datos_ai4i()` busca el CSV en `src/sincronicas/datos/`; si no está, lo descarga de UCI y lo guarda; si no hay internet, genera datos simulados con las mismas reglas.

In [11]:
RUTA_DATOS = ROOT / "src" / "sincronicas" / "datos" / "ai4i2020.csv"
df, origen = cargar_datos_ai4i(RUTA_DATOS)

print("Origen de los datos:", origen)
print("Dimensiones:", df.shape)
df.head()

Origen de los datos: archivo local ai4i2020.csv
Dimensiones: (10000, 14)


,udi,producto_id,tipo,temp_aire_K,temp_proceso_K,velocidad_rpm,torque_Nm,desgaste_min,falla,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [12]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
udi,10000.0,5000.50,2886.90,1.0,2500.75,5000.5,7500.25,10000.0
temp_aire_K,10000.0,300.00,2.00,295.3,298.30,300.1,301.50,304.5
temp_proceso_K,10000.0,310.01,1.48,305.7,308.80,310.1,311.10,313.8
velocidad_rpm,10000.0,1538.78,179.28,1168.0,1423.00,1503.0,1612.00,2886.0
torque_Nm,10000.0,39.99,9.97,3.8,33.20,40.1,46.80,76.6
desgaste_min,10000.0,107.95,63.65,0.0,53.00,108.0,162.00,253.0
falla,10000.0,0.03,0.18,0.0,0.00,0.0,0.00,1.0
TWF,10000.0,0.00,0.07,0.0,0.00,0.0,0.00,1.0
HDF,10000.0,0.01,0.11,0.0,0.00,0.0,0.00,1.0
PWF,10000.0,0.01,0.10,0.0,0.00,0.0,0.00,1.0


## 4.1 El desbalance

> 🎬 **Escena 5 · De los sensores a los datos** — **Momento 2 · Desbalance y modos de falla**

Lo primero en un problema de fallas: ¿cuántas hay?

In [13]:
FrontendRedNeuronal.figura_balance(df)

In [14]:
FrontendRedNeuronal.figura_modos(df)

## 4.2 ¿Dónde aparecen las fallas?

> 🎬 **Escena 5 · De los sensores a los datos** — **Momento 3 · ¿Dónde aparecen las fallas?**
>
> Este es también el **Momento 3 de la Escena 2**: las fallas son combinaciones que ninguna línea recta separa.

Las fallas (rojo) se concentran en los **extremos**: velocidad baja con torque alto, velocidad alta con torque muy bajo. No hay una línea recta que las separe del resto: por eso un modelo lineal se queda corto.

In [15]:
FrontendRedNeuronal.figura_dispersion(df, "velocidad_rpm", "torque_Nm")

In [16]:
FrontendRedNeuronal.figura_dispersion(df, "desgaste_min", "torque_Nm")

---
# 5 · Preparar los datos

## 5.1 Variables físicas (ingeniería de variables)

> 🎬 **Escena 5 · De los sensores a los datos** — **Momento 4 · Variables físicas**

La red podría descubrir sola las combinaciones, pero si se le dan las variables que la **física** del proceso sugiere, aprende más rápido y con menos datos. Es la forma de poner el conocimiento del experto de planta dentro del modelo:

| Variable derivada | Fórmula | Relación con la falla |
|---|---|---|
| `dif_temp_K` | `temp_proceso_K − temp_aire_K` | Disipación de calor (HDF) |
| `potencia_W` | `torque_Nm × velocidad_rpm × 2π / 60` | Potencia fuera de rango (PWF) |
| `esfuerzo_minNm` | `desgaste_min × torque_Nm` | Sobreesfuerzo (OSF) |

## 5.2 Tres conjuntos de datos

> 🎬 **Escena 5 · De los sensores a los datos** — **Momento 5 · Dividir y escalar**

| Conjunto | % | Para qué se usa |
|---|---|---|
| Entrenamiento | 70 % | La red aprende los pesos (aquí se sobremuestrean las fallas) |
| Validación | 15 % | Se elige el **umbral** de decisión según los costos |
| Prueba | 15 % | Evaluación final: simula máquinas nuevas que el modelo nunca vio |

La división es **estratificada**: cada conjunto conserva el ~3.4 % de fallas.

La celda siguiente también fija las **perillas del MLP** (Escena 4 · Momento 4).

In [17]:
modelo = RedNeuronalMantenimiento(
    capas_ocultas=(32, 16),
    activacion="relu",
    alpha=1e-3,
    tasa_aprendizaje=1e-3,
    max_iter=300,
    variables_derivadas=True,
    costo_falla=10_000,
    costo_mantenimiento=600,
)
pd.Series(modelo.preparar(df, origen)).to_frame("registros")

,registros
entrenamiento,7000
validación,1500
prueba,1500
fallas_entrenamiento,237
fallas_validación,51
fallas_prueba,51


In [18]:
print("Variables de entrada:", modelo.variables)
modelo.X_train.head()

Variables de entrada: ['tipo', 'temp_aire_K', 'temp_proceso_K', 'velocidad_rpm', 'torque_Nm', 'desgaste_min', 'dif_temp_K', 'potencia_W', 'esfuerzo_minNm']


,tipo,temp_aire_K,temp_proceso_K,velocidad_rpm,torque_Nm,desgaste_min,dif_temp_K,potencia_W,esfuerzo_minNm
5771,L,301.6,311.2,1525,43.7,91,9.6,6978.786281,3976.7
771,H,296.8,308.0,1311,58.6,19,11.2,8045.053299,1113.4
7708,L,300.5,311.4,1709,32.3,64,10.9,5780.603786,2067.2
1795,M,298.1,307.7,1498,38.3,81,9.6,6008.128398,3102.3
7887,L,300.7,312.4,1590,37.9,98,11.7,6310.517163,3714.2


## 5.3 ¿Por qué escalar?

> 🎬 **Escena 5 · De los sensores a los datos** — **Momento 5 · Dividir y escalar**

Las variables tienen escalas muy distintas. El `StandardScaler` del pipeline las lleva a media 0 y desviación 1. Se ajusta **solo con el entrenamiento** (dentro del `Pipeline`) para no filtrar información de validación o prueba.

In [19]:
numericas = [v for v in modelo.variables if v != "tipo"]
pd.DataFrame({"media": modelo.X_train[numericas].mean(), "desviación": modelo.X_train[numericas].std(),
              "mínimo": modelo.X_train[numericas].min(), "máximo": modelo.X_train[numericas].max()}).round(1)

,media,desviación,mínimo,máximo
temp_aire_K,300.0,2.0,295.3,304.4
temp_proceso_K,310.0,1.5,305.7,313.8
velocidad_rpm,1539.0,180.0,1168.0,2886.0
torque_Nm,40.0,10.0,3.8,76.6
desgaste_min,107.9,63.4,0.0,253.0
dif_temp_K,10.0,1.0,7.6,12.1
potencia_W,6280.4,1068.1,1148.4,10469.9
esfuerzo_minNm,4310.2,2814.9,0.0,14999.5


---
# 6 · Entrenar la red neuronal

> 🎬 **Escena 6 · Entrena y evalúa tu red** — **Momento 2 · Entrenamiento**

`entrenar()` hace, en orden:

1. **Sobremuestrea** las fallas del entrenamiento (las repite al azar hasta que sean ~1/3).
2. Arma el `Pipeline`: `StandardScaler` + `OneHotEncoder(tipo)` → `MLPClassifier`.
3. Entrena la red con Adam durante `max_iter` épocas.
4. Elige el umbral que **minimiza el costo** en validación.

In [20]:
%%time
modelo.entrenar()
print(f"Épocas: {modelo.red.n_iter_} · pérdida final: {modelo.red.loss_:.4f} · umbral elegido: {modelo.umbral:.2f}")

Épocas: 300 · pérdida final: 0.0290 · umbral elegido: 0.17
CPU times: total: 4.45 s
Wall time: 4.63 s


## 6.1 La arquitectura

> 🎬 **Escena 6 · Entrena y evalúa tu red** — **Momento 1 · La arquitectura**

11 entradas (8 numéricas + 3 columnas del tipo) → 32 neuronas → 16 neuronas → 1 salida con sigmoide. Cada conexión es un peso que la red aprendió.

In [21]:
arq = modelo.arquitectura()
display(arq)
print("Total de parámetros aprendidos:", arq["parámetros"].sum())

,capa,neuronas,activación,parámetros
0,Entrada,11,—,0
1,Oculta 1,32,relu,384
2,Oculta 2,16,relu,528
3,Salida,1,logistic (sigmoide),17


Total de parámetros aprendidos: 929


In [22]:
front = FrontendRedNeuronal(modelo)
front.figura_arquitectura()

## 6.2 La curva de aprendizaje

> 🎬 **Escena 6 · Entrena y evalúa tu red** — **Momento 2 · Entrenamiento (la curva de pérdida)**

La pérdida baja rápido al principio y luego se estabiliza: la red ya encontró buenos pesos.

In [23]:
front.figura_curva_perdida()

---
# 7 · Evaluar el modelo

> 🎬 **Escena 6 · Entrena y evalúa tu red** — **Momento 3 · La trampa de la exactitud**

## 7.1 La trampa de la exactitud

Un "modelo" que diga **siempre "no falla"** acierta el ~97 % de las veces... y no anticipa ninguna falla. En problemas desbalanceados la **exactitud engaña**. Las métricas que importan:

| Métrica | Pregunta que responde | En la planta |
|---|---|---|
| **Sensibilidad (recall)** | De las máquinas que fallaron, ¿cuántas anticipó? | Fallas evitadas |
| **Precisión** | De las alertas, ¿cuántas eran fallas reales? | Paradas que valieron la pena |
| **F1** | Equilibrio entre precisión y recall | — |
| **ROC-AUC** | ¿Qué tan bien ordena las máquinas por riesgo? (1 = perfecto, 0.5 = azar) | Calidad del ranking |
| **PR-AUC** | Igual, pero enfocada en la clase rara (falla) | Mejor indicador con desbalance |

In [24]:
y_test = modelo.y_test.to_numpy()
siempre_no = modelo.metricas(y_test, np.zeros(len(y_test)), umbral=0.5)
siempre_no["ROC-AUC"] = siempre_no["PR-AUC"] = np.nan

tabla = pd.DataFrame({
    "Siempre 'no falla'": siempre_no,
    "Red · entrenamiento": modelo.evaluar("entrenamiento"),
    "Red · validación": modelo.evaluar("validación"),
    "Red · prueba": modelo.evaluar("prueba"),
}).T[["umbral", "exactitud", "precisión", "sensibilidad (recall)", "F1", "ROC-AUC", "PR-AUC"]]
tabla.round(3)

,umbral,exactitud,precisión,sensibilidad (recall),F1,ROC-AUC,PR-AUC
Siempre 'no falla',0.500,0.966,0.000,0.000,0.000,NaN,NaN
Red · entrenamiento,0.175,0.978,0.608,1.000,0.756,0.999,0.981
Red · validación,0.175,0.965,0.494,0.765,0.600,0.926,0.720
Red · prueba,0.175,0.962,0.470,0.922,0.623,0.982,0.859


## 7.2 Matriz de confusión (prueba)

> 🎬 **Escena 6 · Entrena y evalúa tu red** — **Momento 4 · Matriz de confusión**

Cada celda es una consecuencia de negocio distinta. Lo más caro es la esquina inferior izquierda: **fallas no anticipadas**.

In [25]:
front.figura_matriz()

In [26]:
front.figura_roc_pr()

## 7.3 ¿La red vale la pena? Comparación con otros modelos

> 🎬 **Escena 6 · Entrena y evalúa tu red** — **Momento 5 · ¿Vale la pena la red?**

Se compara con la **regresión logística** (una sola neurona, frontera lineal) y con la misma red **sin** las variables físicas. Para que sea justo, cada modelo usa su propio umbral óptimo de costo, elegido en validación.

In [27]:
def resumen(nombre, pipeline, red_ref):
    pv = pipeline.predict_proba(red_ref.X_val)[:, 1]
    umbrales = np.round(np.linspace(0.01, 0.99, 99), 2)
    u = min(umbrales, key=lambda t: red_ref.costo(red_ref.y_val, pv, t)["costo_con_modelo"])
    pt = pipeline.predict_proba(red_ref.X_test)[:, 1]
    met, cos = red_ref.metricas(red_ref.y_test, pt, u), red_ref.costo(red_ref.y_test, pt, u)
    return {"modelo": nombre, "umbral": u, "recall": met["sensibilidad (recall)"], "precisión": met["precisión"],
            "ROC-AUC": met["ROC-AUC"], "PR-AUC": met["PR-AUC"], "costo prueba (US$)": cos["costo_con_modelo"],
            "ahorro %": cos["ahorro_pct"]}

sin_fisica = RedNeuronalMantenimiento(variables_derivadas=False)
sin_fisica.preparar(df, origen)
sin_fisica.entrenar()

comparacion = pd.DataFrame([
    resumen("Regresión logística (1 neurona)", modelo.entrenar_referencia(), modelo),
    resumen("Red neuronal sin variables físicas", sin_fisica.pipeline, sin_fisica),
    resumen("Red neuronal con variables físicas", modelo.pipeline, modelo),
]).set_index("modelo")
comparacion.round(3)

,umbral,recall,precisión,ROC-AUC,PR-AUC,costo prueba (US$),ahorro %
modelo,,,,,,,
Regresión logística (1 neurona),0.63,0.784,0.348,0.946,0.463,179000.0,0.649
Red neuronal sin variables físicas,0.83,0.824,0.656,0.976,0.816,128400.0,0.748
Red neuronal con variables físicas,0.17,0.922,0.470,0.982,0.859,100000.0,0.804


**Lectura:** la regresión logística, al ser lineal, no captura las combinaciones de variables que causan las fallas. La red sin variables físicas ya mejora mucho porque aprende esas combinaciones; con las variables físicas lo hace mejor y con menor costo. **Conocimiento del negocio + red neuronal** es la mejor combinación.

## 7.4 Experimento: ¿más neuronas es mejor?

> 🎬 **Escena 6 · Entrena y evalúa tu red** — **Momento 6 · ¿Más neuronas es mejor?**

Se entrena la misma red con cuatro tamaños distintos y se compara el desempeño en entrenamiento y en prueba.

In [28]:
filas = []
for capas in [(4,), (16,), (32, 16), (64, 32, 16)]:
    r = RedNeuronalMantenimiento(capas_ocultas=capas)
    r.preparar(df, origen)
    r.entrenar()
    ev_tr, ev_te = r.evaluar("entrenamiento"), r.evaluar("prueba")
    filas.append({"capas": str(capas), "parámetros": r.arquitectura()["parámetros"].sum(),
                  "PR-AUC entrenamiento": ev_tr["PR-AUC"], "PR-AUC prueba": ev_te["PR-AUC"],
                  "recall prueba": ev_te["sensibilidad (recall)"], "ahorro % prueba": ev_te["ahorro_pct"]})
pd.DataFrame(filas).set_index("capas").round(3)

,parámetros,PR-AUC entrenamiento,PR-AUC prueba,recall prueba,ahorro % prueba
capas,,,,,
"(4,)",53,0.734,0.718,0.902,0.743
"(16,)",209,0.830,0.749,0.961,0.722
"(32, 16)",929,0.981,0.859,0.922,0.804
"(64, 32, 16)",3393,0.999,0.845,0.784,0.710


Una red muy pequeña no tiene capacidad suficiente (**subajuste**). Una muy grande se acerca a la perfección en entrenamiento pero no necesariamente mejora en prueba (**sobreajuste**): más parámetros no siempre es mejor. La arquitectura `(32, 16)` es un buen equilibrio.

**Ejercicio:** cambia `activacion="tanh"`, `alpha=0.01` o `tasa_aprendizaje=0.01` y compara.

---
# 8 · El umbral de decisión según los costos

> 🎬 **Escena 7 · El umbral se elige con dinero** — **Momento 1 · Costo por umbral**

La red entrega una **probabilidad**. La decisión "parar o seguir" necesita un **umbral**. El valor por defecto (0.5) no tiene nada que ver con el negocio: si una falla cuesta 17 veces más que un mantenimiento, conviene parar antes.

Para cada umbral posible se calcula, en **validación**:

$$\text{costo} = \underbrace{FN \times 10\,000}_{\text{fallas no anticipadas}} + \underbrace{(VP + FP) \times 600}_{\text{mantenimientos preventivos}}$$

y se elige el umbral con el menor costo. El conjunto de prueba **no** se usa para elegirlo, solo para medir.

In [29]:
front.figura_costos()

In [30]:
curva = modelo.curva_costos("validación")
curva[curva["umbral"].isin([0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9])][
    ["umbral", "VP", "FN", "FP", "paradas", "costo_con_modelo", "ahorro_pct"]].round(3)

,umbral,VP,FN,FP,paradas,costo_con_modelo,ahorro_pct
4,0.05,39,12,72,111,186600.0,0.634
9,0.10,39,12,54,93,175800.0,0.655
19,0.20,38,13,39,77,176200.0,0.655
29,0.30,38,13,32,70,172000.0,0.663
49,0.50,37,14,28,65,179000.0,0.649
69,0.70,36,15,20,56,183600.0,0.640
89,0.90,33,18,14,47,208200.0,0.592


> 🎬 **Escena 7 · El umbral se elige con dinero** — **Momento 2 · El rigor: 0,06 vs. 0,175**

En teoría, si la probabilidad de la red fuera **exacta**, convendría parar cuando el costo esperado de seguir supera el de parar:

$$p \times 10\,000 > 600 \;\Rightarrow\; p > 0{,}06$$

Pero la red se entrenó con fallas **sobremuestreadas** (~1/3 en vez de ~3 %): sus probabilidades **no están calibradas**. Por eso el umbral no sale de la fórmula: **se elige con datos**, en validación.

In [31]:
umbral_teorico = COSTO_MANTENIMIENTO / COSTO_FALLA
print(f"Umbral teórico (si p fuera exacta): {umbral_teorico:.3f}")
print(f"Umbral elegido en validación:        {modelo.umbral:.3f}")
p_val = modelo.predecir_proba(X=modelo.X_val)
print(f"Fallas reales en validación: {modelo.y_val.mean():.1%} · probabilidad media que da la red: {p_val.mean():.1%}  ← no calibrada")

Umbral teórico (si p fuera exacta): 0.060
Umbral elegido en validación:        0.175
Fallas reales en validación: 3.4% · probabilidad media que da la red: 4.5%  ← no calibrada


## 8.1 Sensibilidad: ¿qué pasa si cambian los costos?

> 🎬 **Escena 7 · El umbral se elige con dinero** — **Momento 3 · Si cambian los costos**

`con_costos()` devuelve una copia del modelo con otros costos y su nuevo umbral óptimo (es lo que hace la barra lateral de la app).

In [32]:
filas = []
for cf in [2_000, 5_000, 10_000, 20_000, 50_000]:
    m = modelo.con_costos(costo_falla=cf, costo_mantenimiento=600)
    r = m.respuesta_negocio()
    filas.append({"costo falla (US$)": cf, "relación": f"{cf / 600:.0f}:1", "umbral óptimo": m.umbral,
                  "% fallas anticipadas": r["% de fallas anticipadas"], "paradas": r["paradas preventivas"],
                  "ahorro (US$)": r["ahorro"], "ahorro %": r["ahorro %"]})
pd.DataFrame(filas).round(3)

,costo falla (US$),relación,umbral óptimo,% fallas anticipadas,paradas,ahorro (US$),ahorro %
0,2000,3:1,0.845,0.824,62,46800.0,0.459
1,5000,8:1,0.325,0.922,87,182800.0,0.717
2,10000,17:1,0.175,0.922,100,410000.0,0.804
3,20000,33:1,0.175,0.922,100,880000.0,0.863
4,50000,83:1,0.020,0.941,139,2316600.0,0.908


Mientras más cara es la falla, **más bajo el umbral**: la planta acepta más paradas preventivas para no dejar escapar fallas.

---
# 9 · Respuesta a la pregunta de negocio

> 🎬 **Escena 7 · El umbral se elige con dinero** — **Momento 4 · La respuesta de negocio**

> ¿Qué máquinas debemos detener para mantenimiento preventivo en el próximo turno, de modo que anticipemos la mayor cantidad de fallas al menor costo total?

In [33]:
r = modelo.respuesta_negocio()
display(Markdown(FrontendRedNeuronal.texto_respuesta(r)))
pd.Series(r).to_frame("valor")

Con el umbral **0.17**, la red anticipa **47 de 51** fallas (**92%**) interviniendo solo el **6.7%** de las máquinas (100 paradas, 53 de ellas innecesarias). El costo pasa de **US$ 510,000** a **US$ 100,000**: un ahorro de **US$ 410,000 (80%)**. **Decisión:** cada turno, detener las máquinas cuya probabilidad supere el umbral, empezando por las de riesgo *Crítico*.

,valor
máquinas evaluadas,1500.000000
fallas reales,51.000000
fallas anticipadas,47.000000
fallas no detectadas,4.000000
paradas preventivas,100.000000
paradas innecesarias,53.000000
% de fallas anticipadas,0.921569
% de máquinas a intervenir,0.066667
costo sin modelo,510000.000000
costo con modelo,100000.000000


## 9.1 ¿Qué fallas anticipa?

> 🎬 **Escena 7 · El umbral se elige con dinero** — **Momento 5 · ¿Qué fallas anticipa?**

Las fallas por calor, potencia y sobreesfuerzo dependen de las variables medidas y la red las detecta casi todas. Las de **desgaste de herramienta** (ocurren al azar entre 200 y 240 min) y las **aleatorias** son más difíciles: no hay una señal clara en los sensores. Para esas, la recomendación complementaria es **cambiar la herramienta antes de los 200 minutos**.

In [34]:
front.figura_deteccion_modos()

In [35]:
modelo.deteccion_por_modo()

,modo,descripción,casos,detectados,% detectado
0,TWF,Desgaste de herramienta,5,2,40.0
1,HDF,Disipación de calor,21,21,100.0
2,PWF,Potencia fuera de rango,11,11,100.0
3,OSF,Sobreesfuerzo,15,15,100.0
4,RNF,Aleatoria,1,0,0.0


## 9.2 La orden de trabajo del turno

> 🎬 **Escena 7 · El umbral se elige con dinero** — **Momento 6 · Orden de trabajo**
>
> En la **Escena 9** esta misma orden se ve turno por turno en el tablero (sección 13).

`priorizar()` recibe las lecturas de las máquinas y devuelve la lista ordenada por riesgo, con la acción recomendada. Esta tabla **es la respuesta operativa** que recibe el jefe de mantenimiento.

In [36]:
turno = front.datos_prueba().sample(200, random_state=7)       # 200 máquinas de un turno
orden = modelo.priorizar(turno.drop(columns="falla"))
print(f"Máquinas a intervenir: {(orden['acción'] == 'Mantenimiento preventivo').sum()} de {len(orden)}")
orden.head(12)

Máquinas a intervenir: 8 de 200


,maquina_id,tipo,temp_aire_K,temp_proceso_K,velocidad_rpm,torque_Nm,desgaste_min,prob_falla,riesgo,acción,costo_esperado_si_opera
0,CNC-8307,M,298.7,310.1,1402,69.7,64,1.0000,Crítico,Mantenimiento preventivo,10000.0
1,CNC-5706,L,302.4,311.9,1290,70.0,139,1.0000,Crítico,Mantenimiento preventivo,10000.0
2,CNC-4080,L,302.0,310.4,1357,58.0,108,1.0000,Crítico,Mantenimiento preventivo,10000.0
3,CNC-0327,L,297.7,308.5,1373,56.7,203,0.9999,Crítico,Mantenimiento preventivo,9999.0
4,CNC-4078,H,302.1,310.7,1294,62.4,101,0.9994,Crítico,Mantenimiento preventivo,9994.0
5,CNC-9414,L,298.1,308.6,1354,53.9,207,0.9974,Crítico,Mantenimiento preventivo,9974.0
6,CNC-4028,L,302.0,310.9,1455,43.2,220,0.9260,Crítico,Mantenimiento preventivo,9260.0
7,CNC-3843,L,302.5,311.2,1227,60.1,167,0.8220,Crítico,Mantenimiento preventivo,8220.0
8,CNC-2324,L,299.3,308.8,1636,29.9,201,0.1661,Medio,Operar,1661.0
9,CNC-8021,M,301.0,312.0,1499,34.6,207,0.1516,Medio,Operar,1516.0


## 9.3 Diagnóstico de una máquina

`predecir_maquina()` es lo que usa la pestaña **🔧 Probar una máquina** de la app. Además de la probabilidad, devuelve **señales de ingeniería** (reglas físicas) que ayudan a explicar por qué la red marcó riesgo.

In [37]:
from clases_po.frontend_red_neuronal import ESCENARIOS

for nombre, valores in ESCENARIOS.items():
    d = modelo.predecir_maquina(**valores)
    print(f"{nombre:42s} → prob {d['probabilidad']:6.1%} · {d['nivel']:8s} · {d['decisión']}")
    for a in d["alertas"]:
        print(f"{'':45s}⚠ {a}")

Operación normal                           → prob   0.0% · Bajo     · Seguir operando y monitorear
Herramienta gastada + torque alto (OSF)    → prob 100.0% · Crítico  · Detener y programar mantenimiento preventivo
                                             ⚠ Herramienta muy desgastada (215 min ≥ 200): programar cambio.
                                             ⚠ Sobreesfuerzo (12,470 min·Nm > 11,000 para tipo L).
Mala disipación de calor (HDF)             → prob 100.0% · Crítico  · Detener y programar mantenimiento preventivo
                                             ⚠ Disipación de calor insuficiente (ΔT 7.5 K < 8.6 y 1330 rpm < 1380).
Potencia baja (PWF)                        → prob 100.0% · Crítico  · Detener y programar mantenimiento preventivo
                                             ⚠ Potencia fuera de rango (2,592 W; normal 3 500–9 000 W).
Potencia excesiva (PWF)                    → prob 100.0% · Crítico  · Detener y programar mantenimiento preventivo
             

---
# 10 · Guardar el modelo

> 🎬 **Escena 8 · Del modelo a la herramienta** — **Momento 3 · Probar y publicar (guardar el modelo)**

La app web carga este archivo. Se guarda en `src/sincronicas/modelos/` (la GitHub Action solo limpia `datos/` y `reportes/`, así que el modelo no se borra).

`guardar()` escribe un diccionario (no el objeto de la clase), por eso el archivo se puede abrir tanto desde el notebook como desde la app. Guarda también la versión de scikit-learn: si no coincide al cargar, la app vuelve a entrenar la red en lugar de fallar.

In [38]:
RUTA_MODELO = ROOT / "src" / "sincronicas" / "modelos" / "red_neuronal_mantenimiento.pkl"
modelo.guardar(RUTA_MODELO)
print(f"Modelo guardado: {RUTA_MODELO.relative_to(ROOT)} ({RUTA_MODELO.stat().st_size / 1024:.0f} KB)")

recargado = RedNeuronalMantenimiento.cargar(RUTA_MODELO)
assert np.allclose(recargado.predecir_proba(X=modelo.X_test), modelo.predecir_proba(X=modelo.X_test))
print("Verificado: el modelo recargado predice exactamente lo mismo.")

Modelo guardado: src\sincronicas\modelos\red_neuronal_mantenimiento.pkl (1094 KB)
Verificado: el modelo recargado predice exactamente lo mismo.


---
# 11 · El frontend en Streamlit: probar, validar y decidir

## 11.1 Qué hace `FrontendRedNeuronal.render()`

> 🎬 **Escena 8 · Del modelo a la herramienta** — **Momento 2 · La app en Streamlit (seis pestañas)**

| Zona | Contenido | Para qué |
|---|---|---|
| Barra lateral | Costo de falla, costo de mantenimiento, umbral (óptimo o manual) | Ajustar los **supuestos del negocio**; todo se recalcula |
| Cabecera | Pregunta de negocio y 5 KPI | La respuesta en 5 segundos |
| 🎯 Respuesta de negocio | Texto con cifras, curva de costos, matriz de confusión, detección por modo | Responder la pregunta |
| 📅 Tablero de mantenimiento | Bitácora simulada (o CSV de la planta) por fecha y turno → semáforo, indicadores del periodo, tendencias de costo y recall, deriva (PSI), orden de trabajo del turno e historia de cada máquina | **Monitorear** la red en operación y decidir cada turno |
| 🔧 Probar una máquina | Formulario con controles + escenarios de ejemplo → medidor, riesgo, decisión y alertas físicas | **Probar** la red con casos concretos |
| 📋 Validar un lote | Subir un CSV (o usar el conjunto de prueba) → ranking, orden de trabajo descargable y, si trae la columna `falla`, métricas contra la realidad | **Validar** la red con datos nuevos |
| 📈 Desempeño | Métricas por conjunto, ROC, PR, curva de pérdida | Auditar el modelo |
| 🧠 Cómo funciona la red | Diagrama de la arquitectura, tabla de capas y activaciones | Explicar la red a quien no es técnico |

Las figuras son las mismas que se vieron en este notebook: la clase de frontend **no recalcula nada por su cuenta**, todo lo pide al modelo.

## 11.2 El punto de entrada

> 🎬 **Escena 8 · Del modelo a la herramienta** — **Momento 1 · Dos clases**: `src/clases_po/app_mantenimiento.py`

La app vive junto a las clases, en `src/clases_po/`, con un nombre propio (`app_mantenimiento.py`) para no pisar el `app.py` del dashboard de la Semana 6. Igual que esa app, agrega su propia carpeta al `sys.path` e importa solo los dos módulos que necesita (no todo `clases_po`, que pide `datasets`).

Si no encuentra el modelo guardado, o fue guardado con otra versión de scikit-learn, lo **entrena al arrancar** (tarda unos segundos).

In [39]:
APP_DIR = ROOT / "src" / "clases_po"
APP = APP_DIR / "app_mantenimiento.py"
print("La app quedará en:", APP.relative_to(ROOT))

La app quedará en: src\clases_po\app_mantenimiento.py


In [40]:
%%writefile ../clases_po/app_mantenimiento.py
"""App web: mantenimiento predictivo con redes neuronales (Semana 7).

Ejecutar en local (desde la raíz del repositorio):
    streamlit run src/clases_po/app_mantenimiento.py

En Streamlit Community Cloud: Main file path = src/clases_po/app_mantenimiento.py
"""
import sys
from pathlib import Path

import streamlit as st

AQUI = Path(__file__).resolve().parent                    # src/clases_po
SRC = AQUI.parent                                         # src/
sys.path.insert(0, str(AQUI))                             # importa los módulos sin cargar todo el paquete

from red_neuronal import RedNeuronalMantenimiento         # noqa: E402
from frontend_red_neuronal import FrontendRedNeuronal     # noqa: E402

RUTA_MODELO = SRC / "sincronicas" / "modelos" / "red_neuronal_mantenimiento.pkl"
RUTA_DATOS = SRC / "sincronicas" / "datos" / "ai4i2020.csv"


@st.cache_resource(show_spinner="Cargando la red neuronal…")
def cargar_modelo() -> RedNeuronalMantenimiento:
    try:
        return RedNeuronalMantenimiento.cargar(RUTA_MODELO)
    except Exception as e:                                # no existe o es de otra versión de scikit-learn
        print(f"No se usó el modelo guardado ({e}). Se entrena de nuevo.")
        return RedNeuronalMantenimiento.entrenar_desde_cero(RUTA_DATOS)


FrontendRedNeuronal(cargar_modelo()).render()

Writing ../clases_po/app_mantenimiento.py


## 11.3 Dependencias de la app: `src/clases_po/requirements.txt`

Streamlit Cloud busca primero el `requirements.txt` en la carpeta del archivo principal. Como `src/clases_po/` también aloja el dashboard de la Semana 6, el archivo incluye lo que necesitan **las dos apps**. Se fija la **misma versión de scikit-learn** con la que se guardó el modelo para que el `.pkl` cargue sin reentrenar.

In [41]:
import sklearn

requisitos = f'''streamlit>=1.50
pandas>=2.2
numpy>=1.26
plotly>=5.24
scikit-learn=={sklearn.__version__}
'''
(APP_DIR / "requirements.txt").write_text(requisitos, encoding="utf-8")
print(requisitos)

streamlit>=1.50
pandas>=2.2
numpy>=1.26
plotly>=5.24
scikit-learn==1.9.1



## 11.4 Prueba automática (sin navegador)

> 🎬 **Escena 8 · Del modelo a la herramienta** — **Momento 3 · Probar y publicar**

`AppTest` ejecuta la app como lo haría Streamlit y permite revisar qué dibujó y simular a un usuario. Si algo falla, aparece aquí antes de publicar.

In [42]:
import logging
from streamlit.testing.v1 import AppTest

logging.getLogger("streamlit").setLevel(logging.ERROR)
at = AppTest.from_file(str(APP), default_timeout=300).run()
print("Errores:", len(at.exception))
print("Pestañas:", [t.label for t in at.tabs])
print("KPI:", [(k.label, k.value) for k in at.metric[:6]])
assert not at.exception, at.exception

2026-11-13 19:08:56.746 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Errores: 0
Pestañas: ['🎯 Respuesta de negocio', '🔧 Probar una máquina', '📋 Validar un lote', '📈 Desempeño del modelo', '🧠 Cómo funciona la red']
KPI: [('Fallas anticipadas', '92%'), ('Máquinas a intervenir', '6.7%'), ('Costo sin modelo', 'US$ 510,000'), ('Costo con la red', 'US$ 100,000'), ('Ahorro', 'US$ 410,000'), ('Máquinas a intervenir', '100')]


Ahora se simula a un usuario: elige el escenario de **sobreesfuerzo** en la pestaña de prueba y luego sube el costo de la falla en la barra lateral.

In [43]:
at.selectbox(key="escenario").set_value("Herramienta gastada + torque alto (OSF)").run()
assert not at.exception, at.exception
print("Decisión mostrada:", [e.value for e in at.error] + [s.value for s in at.success])
print("Alertas físicas:", [w.value for w in at.warning])

at.sidebar.number_input[0].set_value(30_000).run()
assert not at.exception, at.exception
print("Umbral con falla de US$ 30 000:", [k.value for k in at.sidebar.metric])

Decisión mostrada: ['**Detener y programar mantenimiento preventivo**']
Alertas físicas: ['Herramienta muy desgastada (215 min ≥ 200): programar cambio.', 'Sobreesfuerzo (12,470 min·Nm > 11,000 para tipo L).']
Umbral con falla de US$ 30 000: ['0.17']


Por último, el tablero: se activa la ola de calor y se cambia el turno de la orden de trabajo.

In [44]:
at.checkbox(key="tab_deriva").check().run()
assert not at.exception, at.exception
print("Semáforo:", [e.value[:120] for e in at.error])
at.selectbox(key="tab_turno_ot").set_value("Noche").run()
assert not at.exception, at.exception
print("Orden de trabajo:", [(k.label, k.value) for k in at.metric if k.label in ("Máquinas a intervenir", "Riesgo crítico")])

KeyError: 'tab_deriva'

## 11.5 Abrir la app en el navegador

La celda arranca Streamlit en segundo plano; abre **http://localhost:8502** (el 8501 queda libre para el dashboard de la Semana 6). Para detenerla, ejecuta la celda siguiente.

> También se puede correr desde una terminal, en la raíz del proyecto: `streamlit run src/clases_po/app_mantenimiento.py`

**Cómo validar la red en la app:**
1. Pestaña **🔧**: recorre los escenarios de ejemplo y mueve los controles. Por ejemplo, con el escenario normal, sube el desgaste a 220 min y el torque a 60 Nm: el riesgo debe dispararse.
2. Pestaña **📋**: descarga la plantilla, cámbiale valores y súbela; o sube un CSV con la columna `falla` para comparar contra la realidad.
3. Barra lateral: cambia los costos y observa cómo se mueven el umbral, las paradas y el ahorro.

In [ ]:
import subprocess

ABRIR_EN_NAVEGADOR = False   # ← cámbialo a True para levantar la app desde aquí

servidor = None
if ABRIR_EN_NAVEGADOR:
    servidor = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", str(APP), "--server.port", "8502"], cwd=ROOT)
    print("App en http://localhost:8502")
else:
    print("Para verla: pon ABRIR_EN_NAVEGADOR = True o usa la terminal.")

Para verla: pon ABRIR_EN_NAVEGADOR = True o usa la terminal.


In [ ]:
if servidor is not None:
    servidor.terminate()
    print("App detenida.")

---
# 12 · Publicar la app

> 🎬 **Escena 8 · Del modelo a la herramienta** — **Momento 3 · Probar y publicar**

Se sigue el mismo proceso de la Semana 6.

**1. Subir los archivos a GitHub** (pon `SUBIR_A_GITHUB = True` o hazlo desde *Source Control* de VS Code):

In [ ]:
SUBIR_A_GITHUB = False   # ← cámbialo a True para hacer pull, commit y push

ARCHIVOS = [
    "src/clases_po/red_neuronal.py",
    "src/clases_po/frontend_red_neuronal.py",
    "src/clases_po/__init__.py",
    "src/clases_po/app_sinc6.py",
    "src/clases_po/requirements.txt",
    "src/sincronicas/modelos/red_neuronal_mantenimiento.pkl",
    "src/sincronicas/Semana_7SINC.ipynb",
]

def git(*args):
    r = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True)
    print(f"$ git {' '.join(args)}\n{(r.stdout + r.stderr).strip()}\n")
    return r.returncode

git("status", "--short")

if SUBIR_A_GITHUB:
    git("pull", "--rebase", "--autostash", "origin", "main")
    git("add", *ARCHIVOS)
    git("commit", "-m", "Semana 7: red neuronal de mantenimiento predictivo y app Streamlit")
    git("push", "origin", "HEAD:main")

$ git status --short
M README.md
 M src/clases_po/__init__.py
?? src/clases_po/analisis.py
?? src/clases_po/app_mantenimiento.py
?? src/clases_po/app_sinc6.py
?? src/clases_po/clasificador_text.py
?? src/clases_po/dashboard_streamlit.py
?? src/clases_po/enriquecimiento.py
?? src/clases_po/frontend_red_neuronal.py
?? src/clases_po/graficos.py
?? src/clases_po/llm_nvidia.py
?? src/clases_po/preprocesamiento.py
?? src/clases_po/red_neuronal.py
?? src/clases_po/requirements.txt
?? src/sincronicas/Semana_3SINC_ML_clasificacion.ipynb
?? src/sincronicas/Semana_4SINC_PI3_Pizzeria_Multisede_.ipynb
?? src/sincronicas/Semana_4SINC_riesgo_credito.ipynb
?? src/sincronicas/Semana_5SINC_ML_KMEANTS_perfilamiento_clientes.ipynb
?? src/sincronicas/Semana_6SINC_ML_REDNEURONAL_STREAMLIT.ipynb
?? src/sincronicas/Semana_6SINC_PI3_STEAMLIT.ipynb
?? src/sincronicas/datos/ai4i2020.csv
?? src/sincronicas/datos/car_price_ml_regressionlineal.csv
?? src/sincronicas/datos/credit_card_clients.csv
?? src/sincronicas/

**2. Crear la app en https://share.streamlit.io** → *Create app* → *Deploy a public app from GitHub*:

| Campo | Valor |
|---|---|
| Repository | tu repositorio del curso |
| Branch | `main` |
| Main file path | `src/clases_po/app_mantenimiento.py` |
| App URL (opcional) | por ejemplo `mantenimiento-predictivo-ml` |

En *Advanced settings* elige **Python 3.12**. Pega la URL aquí:

In [ ]:
URL_APP = ""   # ← por ejemplo "https://mantenimiento-predictivo-ml.streamlit.app"

if URL_APP:
    display(Markdown(f"### App publicada: [{URL_APP}]({URL_APP})"))
else:
    print("Aún no hay URL: sigue los pasos de la sección 12.")

Aún no hay URL: sigue los pasos de la sección 12.


> 🎬 **Escena 8 · Del modelo a la herramienta** — **Momento 4 · Límites honestos**

- **Supuesto optimista:** el mantenimiento preventivo no siempre evita la falla.
- **Datos sintéticos:** AI4I fue generado para investigación, con reglas de procesos reales.
- **Solo sabe lo que vio:** si la planta cambia (otra estación, otro proveedor de herramientas, un sensor descalibrado), la red puede equivocarse sin que el código cambie.

Por eso la red hay que **vigilarla todos los días**. Para eso está el tablero.

---
# 13 · El tablero responde la pregunta

> 🎬 **Escena 9 · El tablero responde la pregunta** — 7 momentos

Volvemos a la pregunta de la sección 0, ahora con la pestaña **📅 Tablero de mantenimiento** de la app. Cada momento abre uno de los candados:

| Momento | Zona del tablero | 🔓 Pregunta que responde | Método del modelo |
|---|---|---|---|
| 1 · Fuente de datos | ⚙️ Fuente de datos del tablero | ¿De dónde salen los datos y cada cuánto se actualizan? | `simular_operacion()`, `monitorear()` |
| 2 · Periodo y turnos | Filtros *Periodo* y *Turnos* | ¿Qué ventana evaluamos? ¿Qué turno carga más? | `resumen_monitoreo(..., "turno")` |
| 3 · Indicadores | 📊 Indicadores del periodo | ¿Cuántas fallas anticipamos? ¿Cuánto ahorramos? | `resumen_monitoreo()` |
| 4 · Personal | 👷 Personal de mantenimiento | ¿Cuántas máquinas y cuántos técnicos por turno? | `carga_personal()` |
| 5 · Monitoreo | 🚦 Estado del modelo y 🌡️ Monitoreo de los datos | ¿Siguen iguales los datos? | `deriva()` |
| 6 · Orden de trabajo | 🧾 Orden de trabajo del turno | ¿Cuál es la orden de hoy? | `orden_trabajo()` |
| 7 · Cierre | — | La pregunta principal | todas |

## 13.1 La fuente de datos y cada cuánto se actualiza

> 🎬 **Escena 9 · El tablero responde la pregunta** — **Momento 1 · Fuente de datos**

**Cómo corre el modelo en la planta:**

```
sensores ──► bitácora del turno ──► puntuar(): la red da una probabilidad ──► umbral por costo ──► riesgo y acción ──► tablero
 (5 lecturas)   (fecha · turno · máquina)        (segundos)                       (0,175)            (Crítico / Alto / …)
```

| ¿Cada cuánto? | Qué se hace |
|---|---|
| **Cada turno** | Se carga la bitácora del turno que termina y la red puntúa todas las máquinas: la orden de trabajo sale al iniciar el turno siguiente |
| **Cada mes, o cuando el monitoreo detecta deriva** | Se reentrena la red con datos recientes y se guarda un modelo nuevo (`guardar()`) |

En clase no hay una bitácora real, así que `simular_operacion()` crea una: **200 fresadoras, 3 turnos por día, 14 días**. Es una simulación de ciclo cerrado: la red puntúa cada turno y, si una máquina recibe mantenimiento preventivo (o falla, o la herramienta llega a 240 min), su herramienta se cambia y el desgaste vuelve a 0. Las fallas se marcan con las reglas documentadas de AI4I 2020. En la planta se usa `monitorear()` con el CSV real (columnas `fecha`, `turno`, `maquina_id`, `tipo` y los 5 sensores).

In [ ]:
bitacora = modelo.simular_operacion(dias=14, maquinas=200)
print(f"Lecturas: {len(bitacora):,} = 14 días × 3 turnos × 200 máquinas")
print("Columnas que agrega la red:", ["prob_falla", "riesgo", "acción", "costo_esperado_si_opera"])
bitacora[["fecha", "turno", "maquina_id", "tipo", "desgaste_min", "torque_Nm", "prob_falla", "riesgo", "acción"]].head()

Lecturas: 8,400 = 14 días × 3 turnos × 200 máquinas
Columnas que agrega la red: ['prob_falla', 'riesgo', 'acción', 'costo_esperado_si_opera']


,fecha,turno,maquina_id,tipo,desgaste_min,torque_Nm,prob_falla,riesgo,acción
0,2026-10-31,Mañana,CNC-0001,M,128,50.8,0.0000,Bajo,Operar
1,2026-10-31,Mañana,CNC-0002,H,162,55.1,0.0000,Bajo,Operar
2,2026-10-31,Mañana,CNC-0003,M,87,67.9,0.1413,Medio,Operar
3,2026-10-31,Mañana,CNC-0004,L,131,38.2,0.0000,Bajo,Operar
4,2026-10-31,Mañana,CNC-0005,L,43,33.9,0.0000,Bajo,Operar


In [ ]:
import io
# Así se usaría con la bitácora real de la planta (aquí, el CSV de ejemplo que también descarga la app)
bitacora_planta = pd.read_csv(io.BytesIO(front.bitacora_ejemplo()))
modelo.monitorear(bitacora_planta).head()

,fecha,turno,maquina_id,tipo,temp_aire_K,temp_proceso_K,velocidad_rpm,torque_Nm,desgaste_min,prob_falla,riesgo,acción,costo_esperado_si_opera
0,2026-11-12,Mañana,CNC-0001,M,298.9,308.9,1497,46.8,53,0.0000,Bajo,Operar,0.0
1,2026-11-12,Mañana,CNC-0002,H,298.4,307.9,1485,42.0,99,0.0000,Bajo,Operar,0.0
2,2026-11-12,Mañana,CNC-0003,M,299.3,308.5,1393,48.4,75,0.0000,Bajo,Operar,0.0
3,2026-11-12,Mañana,CNC-0004,L,302.9,312.5,1590,34.4,49,0.0000,Bajo,Operar,0.0
4,2026-11-12,Mañana,CNC-0005,L,300.6,312.0,1730,27.5,198,0.0299,Bajo,Operar,299.0


## 13.2 El periodo y los turnos

> 🎬 **Escena 9 · El tablero responde la pregunta** — **Momento 2 · Periodo y turnos**

- **El periodo** es la ventana para evaluar. Un día dice poco (pocas fallas, mucho ruido); una semana o un mes muestran **tendencias**. Con las fechas se compara un "antes y después": un cambio de proveedor, una ola de calor, un modelo reentrenado.
- **Los turnos** definen cada cuánto se mira cada máquina. Con **un solo turno** de lectura, cada máquina se revisa una vez al día y pasa 24 horas sin mirarse: la falla puede llegar entre lecturas. Con **tres turnos**, se revisa cada 8 horas: más lecturas, más oportunidades de llegar antes que la falla.
- El filtro de turnos responde: **¿qué turno carga más mantenimiento?**

> La simulación registra las fallas solo en el momento de cada lectura; por eso sirve para ver la carga por turno, pero no para medir cuántas fallas se escapan *entre* lecturas con menos turnos. Esa comparación requiere la bitácora real.

In [ ]:
ultima_semana = bitacora[bitacora["fecha"] >= bitacora["fecha"].max() - pd.Timedelta(days=6)]
print(f"Periodo: {ultima_semana['fecha'].min()} a {ultima_semana['fecha'].max()}")
modelo.resumen_monitoreo(ultima_semana, "turno")[["turno", "lecturas", "intervenciones", "% a intervenir",
                                                   "fallas", "anticipadas", "recall", "ahorro"]].round(3)

Periodo: 2026-11-07 a 2026-11-13


,turno,lecturas,intervenciones,% a intervenir,fallas,anticipadas,recall,ahorro
0,Mañana,1400,176,0.126,73,66,0.904,554400.0
1,Tarde,1400,165,0.118,68,54,0.794,441000.0
2,Noche,1400,156,0.111,67,59,0.881,496400.0


## 13.3 Indicadores del periodo: ¿cómo nos va?

> 🎬 **Escena 9 · El tablero responde la pregunta** — **Momento 3 · Indicadores del periodo**

Los mismos indicadores de la respuesta de negocio (sección 9), ahora **día por día**. Si la línea de fallas anticipadas cae por debajo del recall de la prueba, **algo cambió**.

In [ ]:
por_dia = modelo.resumen_monitoreo(bitacora, "fecha")
total = por_dia.sum(numeric_only=True)
print(f"Lecturas: {int(total['lecturas']):,} · a intervenir: {int(total['intervenciones']):,} ({total['intervenciones'] / total['lecturas']:.1%})")
print(f"Fallas anticipadas: {int(total['anticipadas'])} de {int(total['fallas'])} ({total['anticipadas'] / total['fallas']:.0%})")
print(f"Costo sin modelo: US$ {total['costo sin modelo']:,.0f} · con la red: US$ {total['costo con modelo']:,.0f} · "
      f"ahorro: US$ {total['ahorro']:,.0f} ({total['ahorro'] / total['costo sin modelo']:.0%})")
front.figura_tendencia_costos(por_dia)

Lecturas: 8,400 · a intervenir: 970 (11.5%)
Fallas anticipadas: 363 de 418 (87%)
Costo sin modelo: US$ 4,180,000 · con la red: US$ 1,132,000 · ahorro: US$ 3,048,000 (73%)


In [ ]:
recall_prueba = modelo.evaluar("prueba")["sensibilidad (recall)"]
front.figura_tendencia_deteccion(por_dia, recall_prueba)

## 13.4 Personal de mantenimiento: ¿cuánta gente muevo?

> 🎬 **Escena 9 · El tablero responde la pregunta** — **Momento 4 · Personal de mantenimiento**

$$\text{técnicos} = \left\lceil \frac{\text{intervenciones} \times \text{horas por intervención}}{\text{horas del turno}} \right\rceil$$

Con 1,5 h por intervención y turnos de 8 h, 23 intervenciones piden ⌈4,3⌉ = **5 técnicos**. Si faltan técnicos, la orden de trabajo ya viene ordenada por riesgo: **primero lo crítico**, y lo demás se reprograma. En la app, las horas y los técnicos disponibles se ajustan en la sección 👷.

In [ ]:
carga = modelo.carga_personal(bitacora, tecnicos_disponibles=5, horas_intervencion=1.5, horas_turno=8)
pico = carga.loc[carga["técnicos necesarios"].idxmax()]
print(f"Técnicos por turno: promedio {carga['técnicos necesarios'].mean():.1f} · pico {pico['técnicos necesarios']} ({pico['fecha']}, {pico['turno']})")
print(f"Turnos con déficit (5 técnicos disponibles): {(carga['déficit'] > 0).sum()} de {len(carga)}")
front.figura_tecnicos(carga)

Técnicos por turno: promedio 4.8 · pico 9 (2026-11-01, Mañana)
Turnos con déficit (5 técnicos disponibles): 11 de 42


In [ ]:
front.figura_alertas_turno(modelo.resumen_monitoreo(bitacora, ["fecha", "turno"]))

## 13.5 Monitoreo de los datos: ¿siguen iguales?

> 🎬 **Escena 9 · El tablero responde la pregunta** — **Momento 5 · Monitoreo de los datos**

**PSI (Population Stability Index):** mide cuánto cambió la distribución de cada variable frente al entrenamiento. Menos de 0,10 es estable, entre 0,10 y 0,25 hay que vigilar, más de 0,25 es **deriva**.

Se simula una **ola de calor**: desde el día 8 la planta tiene 3 K más en el aire y en el proceso. La diferencia de temperatura no cambia, pero la red **nunca vio** temperaturas tan altas.

In [ ]:
bitacora_calor = modelo.simular_operacion(dias=14, deriva_temp_K=3, deriva_desde_dia=7)
psi_calor = modelo.deriva(bitacora_calor)
r_calor = modelo.resumen_monitoreo(bitacora_calor, "fecha")
antes, despues = r_calor.iloc[:7], r_calor.iloc[7:]
print(f"Fallas anticipadas · días 1–7: {antes['anticipadas'].sum() / antes['fallas'].sum():.0%} → "
      f"días 8–14: {despues['anticipadas'].sum() / despues['fallas'].sum():.0%}")
front.figura_deriva(psi_calor)

Fallas anticipadas · días 1–7: 88% → días 8–14: 62%


**Lectura:** las temperaturas se ponen en rojo desde el día 8 y las fallas anticipadas caen: hay que revisar los sensores y **reentrenar** con datos recientes. El **desgaste** en amarillo, en cambio, es un efecto esperado: el propio mantenimiento cambia las herramientas antes de que se gasten. Por eso el tablero no lo cuenta como alerta de deriva (`VARIABLES_CONTROLADAS`).

## 13.6 La orden de trabajo del turno

> 🎬 **Escena 9 · El tablero responde la pregunta** — **Momento 6 · Orden de trabajo del turno**

Se elige fecha y turno, y `orden_trabajo()` devuelve las máquinas a intervenir de mayor a menor riesgo, con el **motivo físico** de cada una. Es lo que se descarga en CSV y se entrega al equipo.

In [ ]:
ultimo_dia = bitacora["fecha"].max()
orden_turno = modelo.orden_trabajo(bitacora, fecha=ultimo_dia, turno="Mañana")
lecturas_turno = bitacora[(bitacora["fecha"] == ultimo_dia) & (bitacora["turno"] == "Mañana")]
print(f"{ultimo_dia} · turno Mañana: {len(orden_turno)} de {len(lecturas_turno)} máquinas a intervenir · "
      f"{lecturas_turno['riesgo'].eq('Crítico').sum()} críticas · "
      f"{modelo.tecnicos_necesarios(len(orden_turno))} técnicos · "
      f"se evitan US$ {orden_turno['costo_esperado_si_opera'].sum():,.0f} en fallas esperadas")
orden_turno.head(10)

2026-11-13 · turno Mañana: 23 de 200 máquinas a intervenir · 21 críticas · 5 técnicos · se evitan US$ 193,999 en fallas esperadas


,fecha,turno,maquina_id,tipo,prob_falla,riesgo,motivo,costo_esperado_si_opera,temp_aire_K,temp_proceso_K,velocidad_rpm,torque_Nm,desgaste_min
0,2026-11-13,Mañana,CNC-0152,H,1.0000,Crítico,Herramienta muy desgastada · Sobreesfuerzo,10000.0,296.7,307.9,1391,54.4,247
1,2026-11-13,Mañana,CNC-0058,L,1.0000,Crítico,Herramienta muy desgastada · Sobreesfuerzo,10000.0,300.6,310.6,1333,61.0,252
2,2026-11-13,Mañana,CNC-0057,H,1.0000,Crítico,Herramienta muy desgastada,10000.0,300.3,309.4,1349,50.4,237
3,2026-11-13,Mañana,CNC-0186,L,0.9998,Crítico,Herramienta muy desgastada · Sobreesfuerzo,9998.0,301.7,310.5,1522,43.8,253
4,2026-11-13,Mañana,CNC-0007,L,0.9990,Crítico,Herramienta muy desgastada · Sobreesfuerzo,9990.0,301.8,310.8,1456,43.6,253
5,2026-11-13,Mañana,CNC-0011,L,0.9984,Crítico,Herramienta muy desgastada · Sobreesfuerzo,9984.0,297.1,307.9,1474,43.7,253
6,2026-11-13,Mañana,CNC-0047,L,0.9790,Crítico,Herramienta muy desgastada,9790.0,302.6,312.1,1439,42.5,253
7,2026-11-13,Mañana,CNC-0127,L,0.9710,Crítico,Herramienta muy desgastada,9710.0,298.1,308.0,1532,42.5,246
8,2026-11-13,Mañana,CNC-0017,H,0.9638,Crítico,Herramienta muy desgastada,9638.0,301.1,310.6,1551,41.2,212
9,2026-11-13,Mañana,CNC-0096,M,0.9586,Crítico,Herramienta muy desgastada,9586.0,299.9,311.3,1484,37.0,253


## 13.7 Cierre: volver a la pregunta

> 🎬 **Escena 9 · El tablero responde la pregunta** — **Momento 7 · Cierre**

> **¿Qué máquinas detener en el próximo turno para anticipar la mayor cantidad de fallas al menor costo?**

La celda siguiente abre los seis candados de la sección 0 con las cifras de esta ejecución.

In [ ]:
rn = modelo.respuesta_negocio()
respuestas = pd.DataFrame([
    ("¿Cuántas fallas anticipamos?", f"{rn['% de fallas anticipadas']:.0%} en la prueba · "
                                    f"{total['anticipadas'] / total['fallas']:.0%} en los 14 días del tablero"),
    ("¿Cuánto ahorramos?", f"US$ {rn['ahorro']:,.0f} en la prueba ({rn['ahorro %']:.0%}) · US$ {total['ahorro']:,.0f} en 14 días"),
    ("¿Cuántas máquinas por turno?", f"≈ {carga['intervenciones'].mean():.0f} de 200"),
    ("¿Cuántos técnicos necesito?", f"≈ {carga['técnicos necesarios'].mean():.0f} por turno · pico de {carga['técnicos necesarios'].max()}"),
    ("¿Siguen iguales los datos?", "el PSI detectó la ola de calor desde el día 8"),
    ("¿Cuál es la orden de hoy?", f"{len(orden_turno)} máquinas, las críticas primero"),
], columns=["🔓 pregunta", "respuesta"])
display(respuestas)
print("\nRespuesta a la pregunta principal: en cada turno, detén las máquinas que la red marca por encima del umbral,")
print(f"empezando por las críticas: anticipas {rn['fallas anticipadas']} de {rn['fallas reales']} fallas y el costo pasa de "
      f"US$ {rn['costo sin modelo']:,.0f} a US$ {rn['costo con modelo']:,.0f}.")

,🔓 pregunta,respuesta
0,¿Cuántas fallas anticipamos?,92% en la prueba · 87% en los 14 días del tablero
1,¿Cuánto ahorramos?,"US$ 410,000 en la prueba (80%) · US$ 3,048,000..."
2,¿Cuántas máquinas por turno?,≈ 23 de 200
3,¿Cuántos técnicos necesito?,≈ 5 por turno · pico de 9
4,¿Siguen iguales los datos?,el PSI detectó la ola de calor desde el día 8
5,¿Cuál es la orden de hoy?,"23 máquinas, las críticas primero"



Respuesta a la pregunta principal: en cada turno, detén las máquinas que la red marca por encima del umbral,
empezando por las críticas: anticipas 47 de 51 fallas y el costo pasa de US$ 510,000 a US$ 100,000.


**Los datos no traían la respuesta. La pregunta sí traía el camino:** pregunta → datos → modelo → umbral → tablero → decisión.

---
# 14 · Conclusiones

1. **Una red neuronal** es una composición de neuronas (`suma ponderada + activación`) que aprende sus pesos con *backpropagation* y descenso del gradiente. Las capas ocultas le permiten aprender **relaciones no lineales** que un modelo lineal no captura (XOR, fallas por combinación de variables).
2. En la industria, el **desbalance** es la norma: la exactitud engaña; se mide con **recall, precisión, PR-AUC** y, sobre todo, **costo**.
3. El **umbral** no es 0.5: se elige con los costos del negocio, en validación, y se evalúa en prueba.
4. El **conocimiento del dominio** (variables físicas) mejora a la red: el experto de planta y el científico de datos trabajan juntos.
5. Un modelo solo crea valor cuando alguien lo **usa**: el frontend en Streamlit convierte la red en una herramienta que el jefe de mantenimiento puede probar, validar con sus propios datos y usar para decidir cada turno.

**Limitaciones:** el supuesto de que el mantenimiento preventivo evita siempre la falla es optimista; las fallas aleatorias y las de desgaste de herramienta son difíciles de anticipar con estos sensores; los datos de AI4I son sintéticos (generados para investigación), aunque reflejan procesos reales.

## Ejercicios propuestos

1. Entrena la red con `activacion="tanh"` y con `capas_ocultas=(64, 32)`. ¿Mejora el ahorro en prueba?
2. Agrega a la app un control para el costo de **horas extra** por parada y súmalo al costo de mantenimiento.
3. Agrega a `RedNeuronalMantenimiento` una regla de negocio: cualquier máquina con `desgaste_min ≥ 200` se marca para cambio de herramienta, aunque la red diga riesgo bajo. ¿Cuánto mejora la detección de fallas TWF?
4. En la pestaña 📋, sube un CSV solo con máquinas tipo **H**. ¿Cambia la proporción de intervenciones?
5. En la pestaña 🔧, elige *Potencia baja (PWF)* y cambia el tipo de producto de **L** a **H**. La probabilidad cae aunque la regla física sigue alertando: en los datos casi no hay máquinas tipo H con potencia baja. ¿Qué dice esto sobre los límites de una red neuronal (solo sabe lo que vio)?

In [ ]:
print("Resumen para la presentación:\n")
print("Pregunta:", "¿Qué máquinas detener en el próximo turno para anticipar más fallas al menor costo?\n")
for k, v in modelo.respuesta_negocio().items():
    print(f"  {k:28s} {v:,.2f}" if isinstance(v, float) else f"  {k:28s} {v:,}")

Resumen para la presentación:

Pregunta: ¿Qué máquinas detener en el próximo turno para anticipar más fallas al menor costo?

  máquinas evaluadas           1,500
  fallas reales                51
  fallas anticipadas           47
  fallas no detectadas         4
  paradas preventivas          100
  paradas innecesarias         53
  % de fallas anticipadas      0.92
  % de máquinas a intervenir   0.07
  costo sin modelo             510,000.00
  costo con modelo             100,000.00
  ahorro                       410,000.00
  ahorro %                     0.80
  umbral                       0.17
